### Data Generation Methodology

This notebook utilizes a custom, deterministic synthetic data generator to simulate a comprehensive multi-channel digital marketing ecosystem spanning a two-year timeframe (2023–2024). Rather than using completely random noise, the underlying algorithm relies on custom probability distributions, seasonal time-series modifiers, and campaign-specific performance profiles. The data accurately replicates real-world behaviors—including weekend traffic drops, late Q4 e-commerce holiday surges, a 15% year-over-year growth trend, and distinct baseline conversion rates across different media channels. Paid channels adaptively calculate variable auction pricing models, while organic channels remain completely isolated from financial expenditures. This yields a robust, structurally sound dataset ideal for high-fidelity performance reporting and advanced business intelligence dashboarding.

### Dataset Structure & Architecture Breakdown

| Dimension/Metric Name | Data Type | Behavioral Logic Included |
| :--- | :--- | :--- |
| **Date / Year / Month** | Temporal | Maps exactly 731 unique dates from Jan 1, 2023 to Dec 31, 2024. |
| **Channel** | Categorical | Segmented by Programmatic, Paid Search, Paid Social, and Organic. |
| **Data_Source** | Categorical | Tied to specific channels (e.g., StackAdapt under Programmatic). |
| **Campaign_Name** | Categorical | Simulates strategies like Brand Core, NonBrand, and Retargeting. |
| **Impressions** | Integer | Driven by negative binomial distribution to model market reach variance. |
| **Clicks / CTR** | Int / Float | Aligned to benchmarks (Search ~4.5%, Programmatic ~0.15%). |
| **Spend / CPC** | Float / Float | Set to 0.00 for Organic; calculates auction cost spikes for paid ads. |
| **Conversions / CVR** | Int / Float | Weighted by intent (Retargeting campaigns convert much higher). |
| **Revenue / ROAS** | Float / Float | Derived from a random baseline average of $85.00 per purchase. |


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# 1. Setup Base Parameters
np.random.seed(42)  # Ensures deterministic, reproducible results
start_date = datetime(2023, 1, 1)
end_date = datetime(2024, 12, 31)
date_range = [start_date + timedelta(days=x) for x in range((end_date - start_date).days + 1)]

# Define ecosystem configurations (Channel -> Data Source -> Campaign Names)
campaign_structure = {
    'Programmatic': {
        'Amazon Ad Server': ['AMZ_DSP_Prospecting_Q1', 'AMZ_DSP_Retargeting_AlwaysOn'],
        'StackAdapt': ['SA_Native_Awareness', 'SA_Video_Intent']
    },
    'Paid Search': {
        'Google Ads': ['GG_Search_Brand_Core', 'GG_Search_NonBrand_Generic', 'GG_PerformanceMax_Products'],
        'Bing Ads': ['BG_Search_Brand', 'BG_Search_Competitors']
    },
    'Paid Social': {
        'Facebook Ads': ['FB_Prospecting_Lookalike', 'FB_Retargeting_DPA', 'IG_Stories_Engagement'],
        'LinkedIn Ads': ['LI_B2B_DecisionMakers', 'LI_LeadGen_Whitepaper']
    },
    'Organic': {
        'Google Search Console': ['SEO_Organic_Blog', 'SEO_Organic_ProductPages'],
        'Organic Social Platform': ['SM_LinkedIn_Organic', 'SM_Instagram_Organic']
    }
}

# 2. Performance Matrix Profile (Base CTR, CPC, and CVR characteristics)
profiles = {
    'Programmatic': {'ctr_mu': 0.0015, 'ctr_sigma': 0.0005, 'cpc_mu': 0.45,  'cpc_sigma': 0.10, 'cvr_mu': 0.008, 'cvr_sigma': 0.002},
    'Paid Search':  {'ctr_mu': 0.0450, 'ctr_sigma': 0.0150, 'cpc_mu': 2.50,  'cpc_sigma': 0.60, 'cvr_mu': 0.035, 'cvr_sigma': 0.010},
    'Paid Social':  {'ctr_mu': 0.0120, 'ctr_sigma': 0.0030, 'cpc_mu': 1.10,  'cpc_sigma': 0.25, 'cvr_mu': 0.022, 'cvr_sigma': 0.005},
    'Organic':      {'ctr_mu': 0.0250, 'ctr_sigma': 0.0080, 'cpc_mu': 0.00,  'cpc_sigma': 0.00, 'cvr_mu': 0.018, 'cvr_sigma': 0.004}
}

data_rows = []

# 3. Data Loop with Seasonality & Volatility Modeling
for current_date in date_range:
    day_of_week = current_date.weekday()
    month = current_date.month
    year = current_date.year

    # Seasonality Modifiers (e.g., Weekend drops, Q4 holiday peaks)
    weekend_modifier = 0.75 if day_of_week >= 5 else 1.0
    holiday_modifier = 1.35 if month in [11, 12] else (1.10 if month in [5, 6] else 1.0)
    yoy_growth = 1.15 if year == 2024 else 1.0  # 15% YoY volume trend inflation

    total_modifier = weekend_modifier * holiday_modifier * yoy_growth

    for channel, sources in campaign_structure.items():
        prof = profiles[channel]

        for source, campaigns in sources.items():
            for campaign in campaigns:
                # Performance fragmentation baseline per campaign
                campaign_hash = sum(ord(char) for char in campaign) % 5
                camp_modifier = 0.8 + (campaign_hash * 0.1) # Scales individual campaign strength

                # Base volume calculations
                base_impressions = np.random.negative_binomial(n=20, p=0.0005) if channel != 'Organic' else np.random.negative_binomial(n=15, p=0.0003)
                impressions = int(max(100, base_impressions * total_modifier * camp_modifier))

                # Metric Generation logic (Clipping bounds to avoid mathematical anomalies)
                ctr = np.clip(np.random.normal(prof['ctr_mu'], prof['ctr_sigma']) * (1.1 if "Retargeting" in campaign else 1.0), 0.0005, 0.25)
                clicks = int(np.round(impressions * ctr))

                if clicks > 0:
                    if channel == 'Organic':
                        spend = 0.0
                        cpc = 0.0
                    else:
                        cpc = np.clip(np.random.normal(prof['cpc_mu'], prof['cpc_sigma']), 0.05, 15.0)
                        spend = round(clicks * cpc, 2)

                    cvr = np.clip(np.random.normal(prof['cvr_mu'], prof['cvr_sigma']) * (1.4 if "Retargeting" in campaign or "Brand" in campaign else 0.9), 0.001, 0.15)
                    conversions = int(np.round(clicks * cvr))
                    revenue = round(conversions * np.random.normal(85.0, 15.0), 2) if conversions > 0 else 0.0
                else:
                    clicks, spend, conversions, revenue, cpc = 0, 0.0, 0, 0.0, 0.0

                # Append clean dictionary row
                data_rows.append({
                    'Date': current_date.strftime('%Y-%m-%d'),
                    'Year': year,
                    'Month': month,
                    'Channel': channel,
                    'Data_Source': source,
                    'Campaign_Name': campaign,
                    'Impressions': impressions,
                    'Clicks': clicks,
                    'CTR': round(clicks / impressions if impressions > 0 else 0, 6),
                    'Spend': spend,
                    'CPC': round(cpc, 4),
                    'Conversions': conversions,
                    'CVR': round(conversions / clicks if clicks > 0 else 0, 6),
                    'Revenue': revenue,
                    'ROAS': round(revenue / spend, 2) if spend > 0 else (0.0 if channel != 'Organic' else np.nan)
                })

# 4. Export & Inspection
df = pd.DataFrame(data_rows)
df.to_csv('synthetic_marketing_data_2023_2024.csv', index=False)

print(f"Dataset generated successfully! Shape: {df.shape}")
print(df[['Channel', 'Data_Source', 'Campaign_Name', 'Impressions', 'CTR', 'Spend', 'Conversions']].head())


Dataset generated successfully! Shape: (13158, 15)
        Channel       Data_Source                 Campaign_Name  Impressions  \
0  Programmatic  Amazon Ad Server        AMZ_DSP_Prospecting_Q1        39547   
1  Programmatic  Amazon Ad Server  AMZ_DSP_Retargeting_AlwaysOn        20839   
2  Programmatic        StackAdapt           SA_Native_Awareness        25252   
3  Programmatic        StackAdapt               SA_Video_Intent        32232   
4   Paid Search        Google Ads          GG_Search_Brand_Core        34873   

        CTR    Spend  Conversions  
0  0.001441    27.24            1  
1  0.000720     5.91            0  
2  0.001188    15.29            0  
3  0.001210    16.41            0  
4  0.041551  3961.00           45  
